In [1]:
# ── CELL 1: Imports, Paths, Constants ─────────────────────────
import pandas as pd
import numpy as np
import os
from datetime import datetime

BASE_DIR    = os.path.dirname(os.path.abspath(''))   # use '' in Jupyter
DATA_DIR    = os.path.join(BASE_DIR, 'data')
SCORES_DIR  = os.path.join(DATA_DIR, 'scores')
FUND_DIR    = os.path.join(DATA_DIR, 'fundamentals')
REPORT_DIR  = os.path.join(DATA_DIR, 'reports', 'reversal')

os.makedirs(REPORT_DIR, exist_ok=True)

today_str  = datetime.now().strftime('%Y-%m-%d')
today_file = datetime.now().strftime('%Y%m%d')

# Filter thresholds
MIN_BOTTOM_PROB = 40.0   # Bottom_Rev_Prob must be >= 40%
MIN_RANK        = 6.5    # Sector Score OR Cap Score must be >= 6.5
TOP_N_PER_CAP   = 15     # Top 15 per cap category
CAP_ORDER       = ['Large Cap', 'Mini Large Cap', 'Mid Cap', 'Small Cap']

print("Cell 1 OK — Paths and constants set")
print(f"  SCORES_DIR : {SCORES_DIR}")
print(f"  REPORT_DIR : {REPORT_DIR}")

Cell 1 OK — Paths and constants set
  SCORES_DIR : E:\learn\Project 1 AI Screener\stock-ai-india\data\scores
  REPORT_DIR : E:\learn\Project 1 AI Screener\stock-ai-india\data\reports\reversal


In [2]:
# ── CELL 2: Load Data ─────────────────────────────────────────
import os

TECH_FILE      = os.path.join(SCORES_DIR, 'technical_report_full.csv')
FUND_FILE      = os.path.join(FUND_DIR,   'fundamental_scores_full.csv')
FUND_PREV_FILE = os.path.join(FUND_DIR,   'fundamental_scores_prev.csv')
FUND_METRICS   = os.path.join(FUND_DIR,   'fund_metrics.csv')          # raw scraped data
PREFILT_FILE   = os.path.join(DATA_DIR,   'universe', 'prefilt_passed.csv')

# Load tech report (master output)
tech_df = pd.read_csv(TECH_FILE)
print(f"  Tech report        : {len(tech_df)} stocks")
print(f"  Columns            : {list(tech_df.columns)}")

# Load current fundamental scores
fund_df = pd.read_csv(FUND_FILE)
print(f"\n  Fund scores (curr) : {len(fund_df)} stocks")
print(f"  Columns            : {list(fund_df.columns)}")

# Load previous quarter snapshot (optional — warn if missing)
prev_available = os.path.exists(FUND_PREV_FILE)
if prev_available:
    fund_prev_df = pd.read_csv(FUND_PREV_FILE)
    print(f"\n  Fund scores (prev) : {len(fund_prev_df)} stocks  ✅")
else:
    fund_prev_df = None
    print(f"\n  Fund scores (prev) : NOT FOUND ⚠️")
    print(f"     Copy old fundamental_scores_full.csv to:")
    print(f"     {FUND_PREV_FILE}")
    print(f"     Section 3 (Score & Rank Movers) will be skipped.")

# Load raw scraped metrics (for deep dive — all fundamental fields)
metrics_available = os.path.exists(FUND_METRICS)
if metrics_available:
    metrics_df = pd.read_csv(FUND_METRICS)
    print(f"\n  Raw fund metrics   : {len(metrics_df)} stocks  ✅")
    print(f"  Columns            : {list(metrics_df.columns)}")
else:
    metrics_df = None
    print(f"\n  Raw fund metrics   : NOT FOUND ⚠️")
    print(f"     Section 2 (Deep Dive) will use fund_scores only.")

# Load prefilt for Market_Cap_Cr
prefilt_df = pd.read_csv(PREFILT_FILE)
print(f"\n  Prefilt data       : {len(prefilt_df)} stocks")

# Merge Market_Cap_Cr into fund_df if missing
if 'Market_Cap_Cr' not in fund_df.columns:
    fund_df = fund_df.merge(
        prefilt_df[['Symbol', 'Market_Cap_Cr']], on='Symbol', how='left'
    )
    print(f"  Market_Cap_Cr merged into fund_df ✅")

print("\nCell 2 OK — Data loaded")

  Tech report        : 752 stocks
  Columns            : ['Symbol', 'Sector', 'Fund Score', 'Market Cap Cr', 'Cap Category', 'Current Price', 'RSI', 'ADX', 'MACD Hist', 'Vol Ratio', 'EMA20', 'EMA50', 'EMA200', 'DI_Plus', 'DI_Minus', 'Momentum Score', 'Reversal Score', 'Best Setup', 'Tech Score', 'Tier', 'In Consolidation', 'Consol Days', 'Consol Label', 'Pct to Breakout', 'Consol Volume', 'ML_Prediction', 'ML_Confidence', 'Forecast_25d_Pct', 'Forecast_45d_Pct', 'Forecast_180d_Pct', 'Forecast_25d_Price', 'Forecast_45d_Price', 'Forecast_180d_Price', 'Bottom_Rev_Prob', 'Top_Rev_Prob', 'Bottom_Rev_Flag', 'Top_Rev_Flag', 'Bullish_Cont_Prob', 'Bearish_Cont_Prob', 'Sector Score', 'Cap Score', 'Vol 5D Ratio', 'Breakout Vol', 'Vol Label', 'Vol Inference', 'Sector Trend', 'Sector Detail', 'Weeks_Count', 'First_Breakout_Date']

  Fund scores (curr) : 752 stocks
  Columns            : ['Symbol', 'Sector', 'Historical Score', 'Peer Score', 'Quality Score', 'Promoter Holding %', 'FII + DII %', 'Fina

In [3]:
# ── CELL 3: Compute Sector Score, Cap Score, merge everything ──

CAP_ORDER = ['Large Cap', 'Mini Large Cap', 'Mid Cap', 'Small Cap']

def classify_mcap(mcap_cr):
    try:
        v = float(mcap_cr)
        if v >= 20000: return 'Large Cap'
        elif v >= 5000: return 'Mini Large Cap'
        elif v >= 1000: return 'Mid Cap'
        else:           return 'Small Cap'
    except:
        return 'Small Cap'

fund_df['Cap Category'] = fund_df['Market_Cap_Cr'].apply(classify_mcap)

# ── Sector Score: relative rank within same sector ─────────────
fund_df['Sector Score'] = 0.0
for sector in fund_df['Sector'].unique():
    mask      = fund_df['Sector'] == sector
    max_score = fund_df[mask]['Final Score'].max()
    if max_score > 0:
        fund_df.loc[mask, 'Sector Score'] = (
            fund_df[mask]['Final Score'] / max_score * 10
        ).round(1)

# ── Cap Score: relative rank within same sector + cap ──────────
fund_df['Cap Score'] = 0.0
for sector in fund_df['Sector'].unique():
    for cap in CAP_ORDER:
        mask   = (fund_df['Sector'] == sector) & (fund_df['Cap Category'] == cap)
        subset = fund_df[mask]
        if len(subset) == 0:
            continue
        max_score = subset['Final Score'].max()
        if max_score > 0:
            fund_df.loc[mask, 'Cap Score'] = (
                subset['Final Score'] / max_score * 10
            ).round(1)

# ── Build master working df ────────────────────────────────────
merge_cols = ['Symbol', 'Historical Score', 'Peer Score',
              'Quality Score', 'Promoter Holding %', 'FII + DII %',
              'Final Score', 'Market_Cap_Cr']

work_df = tech_df.copy()

for col in ['Historical Score', 'Peer Score', 'Quality Score',
            'Promoter Holding %', 'FII + DII %',
            'Final Score', 'Market_Cap_Cr']:
    if col in work_df.columns:
        work_df = work_df.drop(columns=[col])

work_df = work_df.merge(fund_df[merge_cols], on='Symbol', how='left')

print(f"  work_df shape      : {work_df.shape}")
print(f"  Sector Score range : {work_df['Sector Score'].min():.1f} – {work_df['Sector Score'].max():.1f}")
print(f"  Cap Score range    : {work_df['Cap Score'].min():.1f} – {work_df['Cap Score'].max():.1f}")
print(f"  Bottom_Rev_Prob    : {work_df['Bottom_Rev_Prob'].notna().sum()} non-null")
print(f"\n  Cap Category distribution:")
print(work_df['Cap Category'].value_counts().to_string())
print(f"\n  Fund Score breakdown:")
print(f"    Historical Score : min={work_df['Historical Score'].min():.1f}  max={work_df['Historical Score'].max():.1f}  mean={work_df['Historical Score'].mean():.1f}")
print(f"    Peer Score       : min={work_df['Peer Score'].min():.1f}  max={work_df['Peer Score'].max():.1f}  mean={work_df['Peer Score'].mean():.1f}")
print(f"    Quality Score    : min={work_df['Quality Score'].min():.1f}  max={work_df['Quality Score'].max():.1f}  mean={work_df['Quality Score'].mean():.1f}")
print(f"    Final Score      : min={work_df['Final Score'].min():.1f}  max={work_df['Final Score'].max():.1f}  mean={work_df['Final Score'].mean():.1f}")

print("\nCell 3 OK — Working dataframe ready")

  work_df shape      : (752, 56)
  Sector Score range : 2.9 – 10.0
  Cap Score range    : 3.2 – 10.0
  Bottom_Rev_Prob    : 752 non-null

  Cap Category distribution:
Cap Category
Large Cap         214
Mid Cap           210
Mini Large Cap    201
Small Cap         127

  Fund Score breakdown:
    Historical Score : min=0.0  max=40.0  mean=20.7
    Peer Score       : min=10.0  max=38.0  mean=23.4
    Quality Score    : min=4.0  max=19.0  mean=13.7
    Final Score      : min=25.8  max=89.0  mean=57.9

Cell 3 OK — Working dataframe ready


In [7]:
# ── CELL 5: Section 1A — ML Confirmed Reversal ────────────────
import pickle

def mcap_str(mcap_cr):
    try:
        v = float(mcap_cr)
        if v >= 100000: return f"Rs{v/100000:.1f}L Cr"
        return f"Rs{v:,.0f} Cr"
    except:
        return "—"

# Load price data for VMax
PRICE_FILE = os.path.join(DATA_DIR, 'prices', 'price_data_full.pkl')
try:
    with open(PRICE_FILE, 'rb') as f:
        price_data = pickle.load(f)
    print(f"  Price data loaded : {len(price_data)} stocks ✅")
except:
    price_data = {}
    print(f"  Price data NOT loaded ⚠️  VMax will show 0.00x")

def get_vmax(symbol):
    try:
        df = price_data.get(symbol)
        if df is None or len(df) < 20:
            return 0.0
        vol_max5  = df['Volume'].iloc[-5:].max()
        vol_20avg = df['Volume'].iloc[-20:].mean()
        return round(vol_max5 / vol_20avg if vol_20avg > 0 else 0.0, 2)
    except:
        return 0.0

print("=" * 65)
print("  REVERSAL HUNTER — Settings")
print("=" * 65)
try:
    top_n = int(input(
        "\n  How many stocks per cap category? [default 10]: "
    ).strip() or 10)
except:
    top_n = 10

# Section 1A filter
reversal_1a = work_df[
    (work_df['Best Setup']       == 'Reversal') &
    (work_df['Bottom_Rev_Prob']  >= MIN_BOTTOM_PROB) &
    (
        (work_df['Sector Score'] >= MIN_RANK) |
        (work_df['Cap Score']    >= MIN_RANK)
    )
].copy()
reversal_1a = reversal_1a.sort_values('Bottom_Rev_Prob', ascending=False)

print(f"\n  1A — ML CONFIRMED REVERSAL")
print(f"  Filter: Best Setup=Reversal | BotProb>={MIN_BOTTOM_PROB}% | Rank>={MIN_RANK}")
print(f"  Total candidates : {len(reversal_1a)}")

if len(reversal_1a) == 0:
    print(f"  No candidates — market likely in strong uptrend.")
else:
    print(f"\n  {'#':<3}  {'Symbol':<8}  {'MCap':>11}  {'Price':>8}  "
          f"{'BotProb':>7}  {'RevScr':>6}  {'SecRnk':>6}  {'CapRnk':>6}  "
          f"{'RSI':>5}  {'ADX':>5}  {'MACD':>7}  {'V5D':>5}  {'VMax':>5}")
    print(f"  {'─'*112}")

    serial = 1
    for cap in CAP_ORDER:
        cap_df = reversal_1a[reversal_1a['Cap Category'] == cap].head(top_n)
        if len(cap_df) == 0:
            continue
        cap_short = {'Large Cap':'L','Mini Large Cap':'ML',
                     'Mid Cap':'M','Small Cap':'S'}.get(cap,'?')
        print(f"\n  [{cap_short}] {cap}")
        for _, row in cap_df.iterrows():
            print(f"  {serial:<3}  "
                  f"{row['Symbol']:<8}  "
                  f"{mcap_str(row['Market Cap Cr']):>11}  "
                  f"Rs{row['Current Price']:>7.2f}  "
                  f"{row['Bottom_Rev_Prob']:>6.1f}%  "
                  f"{row['Reversal Score']:>6.0f}  "
                  f"{row['Sector Score']:>6.1f}  "
                  f"{row['Cap Score']:>6.1f}  "
                  f"{row['RSI']:>5.1f}  "
                  f"{row['ADX']:>5.1f}  "
                  f"{row['MACD Hist']:>+7.3f}  "
                  f"{float(row.get('Vol 5D Ratio', row.get('Vol Ratio', 0)) or 0):>4.2f}x  "
                  f"{get_vmax(row['Symbol']):>4.2f}x")
            serial += 1

    print(f"\n  {'─'*112}")
    print(f"  BotProb=ML reversal prob | RevScr=Tech reversal score | "
          f"SecRnk/CapRnk=Relative rank 0-10 | V5D=5day avg | VMax=peak day")

symbols_1a = set(reversal_1a['Symbol'].tolist())
print(f"\nCell 5 OK — 1A done | {len(symbols_1a)} symbols to exclude from 1B/1C")

  Price data loaded : 752 stocks ✅
  REVERSAL HUNTER — Settings



  How many stocks per cap category? [default 10]:  5



  1A — ML CONFIRMED REVERSAL
  Filter: Best Setup=Reversal | BotProb>=40.0% | Rank>=6.5
  Total candidates : 2

  #    Symbol           MCap     Price  BotProb  RevScr  SecRnk  CapRnk    RSI    ADX     MACD    V5D   VMax
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────

  [L] Large Cap
  1    IRB       Rs25,068 Cr  Rs  22.26    87.7%      55     7.1     8.9   27.9   42.5   +0.444  0.98x  2.11x

  [S] Small Cap
  2    SPITZE       Rs390 Cr  Rs  60.99    72.9%      50     5.7     7.7   20.6   54.6   +4.237  0.40x  0.58x

  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  BotProb=ML reversal prob | RevScr=Tech reversal score | SecRnk/CapRnk=Relative rank 0-10 | V5D=5day avg | VMax=peak day

Cell 5 OK — 1A done | 2 symbols to exclude from 1B/1C


In [8]:
# ── CELL 5B: Section 1B — Technical & Fundamental Watch ───────

print(f"\n{'='*65}")
print(f"  1B — TECHNICAL & FUNDAMENTAL WATCH")
print(f"  Filter: RevScore>=50 | Rank>={MIN_RANK} | Setup!=Momentum | excl 1A")
print(f"{'='*65}")

try:
    top_n_1b = int(input(
        "\n  How many stocks per cap category? [default 10]: "
    ).strip() or 10)
except:
    top_n_1b = 10

reversal_1b = work_df[
    (work_df['Reversal Score']   >= 50) &
    (
        (work_df['Sector Score'] >= MIN_RANK) |
        (work_df['Cap Score']    >= MIN_RANK)
    ) &
    (work_df['Best Setup']       != 'Momentum') &
    (~work_df['Symbol'].isin(symbols_1a))
].copy()
reversal_1b = reversal_1b.sort_values('Reversal Score', ascending=False)

print(f"\n  Total candidates : {len(reversal_1b)}")

if len(reversal_1b) == 0:
    print(f"  No candidates currently.")
else:
    print(f"\n  {'#':<3}  {'Symbol':<8}  {'MCap':>11}  {'Price':>8}  "
          f"{'BotProb':>7}  {'RevScr':>6}  {'SecRnk':>6}  {'CapRnk':>6}  "
          f"{'RSI':>5}  {'ADX':>5}  {'MACD':>7}  {'V5D':>5}  {'VMax':>5}")
    print(f"  {'─'*112}")

    serial = 1
    for cap in CAP_ORDER:
        cap_df = reversal_1b[reversal_1b['Cap Category'] == cap].head(top_n_1b)
        if len(cap_df) == 0:
            continue
        cap_short = {'Large Cap':'L','Mini Large Cap':'ML',
                     'Mid Cap':'M','Small Cap':'S'}.get(cap,'?')
        print(f"\n  [{cap_short}] {cap}")
        for _, row in cap_df.iterrows():
            print(f"  {serial:<3}  "
                  f"{row['Symbol']:<8}  "
                  f"{mcap_str(row['Market Cap Cr']):>11}  "
                  f"Rs{row['Current Price']:>7.2f}  "
                  f"{row['Bottom_Rev_Prob']:>6.1f}%  "
                  f"{row['Reversal Score']:>6.0f}  "
                  f"{row['Sector Score']:>6.1f}  "
                  f"{row['Cap Score']:>6.1f}  "
                  f"{row['RSI']:>5.1f}  "
                  f"{row['ADX']:>5.1f}  "
                  f"{row['MACD Hist']:>+7.3f}  "
                  f"{float(row.get('Vol 5D Ratio', row.get('Vol Ratio', 0)) or 0):>4.2f}x  "
                  f"{get_vmax(row['Symbol']):>4.2f}x")
            serial += 1

    print(f"\n  {'─'*112}")
    print(f"  RevScr=Technical reversal score | BotProb=ML prob | "
          f"SecRnk/CapRnk=Relative rank 0-10 | V5D=5day avg | VMax=peak day")

symbols_1b = set(reversal_1b['Symbol'].tolist())
print(f"\nCell 5B OK — 1B done | {len(symbols_1b)} candidates")


  1B — TECHNICAL & FUNDAMENTAL WATCH
  Filter: RevScore>=50 | Rank>=6.5 | Setup!=Momentum | excl 1A



  How many stocks per cap category? [default 10]:  5



  Total candidates : 1

  #    Symbol           MCap     Price  BotProb  RevScr  SecRnk  CapRnk    RSI    ADX     MACD    V5D   VMax
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────

  [L] Large Cap
  1    INDUSTOWER    Rs1.1L Cr  Rs 412.25     0.0%      55    10.0    10.0   38.8   20.2   -0.774  1.65x  1.99x

  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  RevScr=Technical reversal score | BotProb=ML prob | SecRnk/CapRnk=Relative rank 0-10 | V5D=5day avg | VMax=peak day

Cell 5B OK — 1B done | 1 candidates


In [10]:
# ── CELL 5C: Section 1C — Basing Watch ───────────────────────

print(f"\n{'='*65}")
print(f"  1C — BASING WATCH")
print(f"  Filter: Price<EMA50 | Rank>={MIN_RANK} | Setup=Watching | ADX<30 | RSI>35")
print(f"  Catches stocks that had reversal signal but faded — still below EMA50")
print(f"{'='*65}")

try:
    top_n_1c = int(input(
        "\n  How many stocks per cap category? [default 10]: "
    ).strip() or 10)
except:
    top_n_1c = 10

reversal_1c = work_df[
    (work_df['Current Price']    <  work_df['EMA50']) &
    (
        (work_df['Sector Score'] >= MIN_RANK) |
        (work_df['Cap Score']    >= MIN_RANK)
    ) &
    (work_df['Best Setup']       == 'Watching') &
    (work_df['ADX']              <  30) &
    (work_df['RSI']              >  35) &
    (~work_df['Symbol'].isin(symbols_1a)) &
    (~work_df['Symbol'].isin(symbols_1b))
].copy()

# Sort by combined rank score descending
reversal_1c['_rank_sum'] = (
    reversal_1c['Sector Score'] + reversal_1c['Cap Score']
)
reversal_1c = reversal_1c.sort_values('_rank_sum', ascending=False)

print(f"\n  Total candidates : {len(reversal_1c)}")

if len(reversal_1c) == 0:
    print(f"  No candidates currently.")
else:
    # EMA50% — how far below EMA50 the price is
    reversal_1c['EMA50_pct'] = (
        (reversal_1c['Current Price'] - reversal_1c['EMA50'])
        / reversal_1c['EMA50'] * 100
    ).round(1)

    print(f"\n  {'#':<3}  {'Symbol':<8}  {'MCap':>11}  {'Price':>8}  "
          f"{'EMA50%':>7}  {'RevScr':>6}  {'SecRnk':>6}  {'CapRnk':>6}  "
          f"{'RSI':>5}  {'ADX':>5}  {'V5D':>5}  {'VMax':>5}")
    print(f"  {'─'*108}")

    serial = 1
    for cap in CAP_ORDER:
        cap_df = reversal_1c[reversal_1c['Cap Category'] == cap].head(top_n_1c)
        if len(cap_df) == 0:
            continue
        cap_short = {'Large Cap':'L','Mini Large Cap':'ML',
                     'Mid Cap':'M','Small Cap':'S'}.get(cap,'?')
        print(f"\n  [{cap_short}] {cap}")
        for _, row in cap_df.iterrows():
            print(f"  {serial:<3}  "
                  f"{row['Symbol']:<8}  "
                  f"{mcap_str(row['Market Cap Cr']):>11}  "
                  f"Rs{row['Current Price']:>7.2f}  "
                  f"{row['EMA50_pct']:>+6.1f}%  "
                  f"{row['Reversal Score']:>6.0f}  "
                  f"{row['Sector Score']:>6.1f}  "
                  f"{row['Cap Score']:>6.1f}  "
                  f"{row['RSI']:>5.1f}  "
                  f"{row['ADX']:>5.1f}  "
                  f"{float(row.get('Vol 5D Ratio', row.get('Vol Ratio', 0)) or 0):>4.2f}x  "
                  f"{get_vmax(row['Symbol']):>4.2f}x")
            serial += 1

    print(f"\n  {'─'*108}")
    print(f"  EMA50% = how far below EMA50 (negative = below) | "
          f"RevScr=Tech score | ADX<30 trend fading | RSI>35 stabilizing")

symbols_1c = set(reversal_1c['Symbol'].tolist())
print(f"\nCell 5C OK — 1C done | {len(symbols_1c)} candidates")


  1C — BASING WATCH
  Filter: Price<EMA50 | Rank>=6.5 | Setup=Watching | ADX<30 | RSI>35
  Catches stocks that had reversal signal but faded — still below EMA50



  How many stocks per cap category? [default 10]:  20



  Total candidates : 50

  #    Symbol           MCap     Price   EMA50%  RevScr  SecRnk  CapRnk    RSI    ADX    V5D   VMax
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────

  [L] Large Cap
  1    3MINDIA   Rs36,352 Cr  Rs31410.00    -4.3%      23    10.0    10.0   46.1   23.8  0.55x  0.71x
  2    UNOMINDA  Rs61,610 Cr  Rs1097.80    -1.4%       8     9.3     9.5   52.2   15.9  1.12x  1.53x
  3    AJANTPHARM  Rs36,308 Cr  Rs2790.20    -2.1%      38     8.3     9.7   44.7   14.1  0.97x  1.56x
  4    HAVELLS   Rs80,369 Cr  Rs1306.40    -0.4%      16     8.3     8.9   55.0   16.6  0.73x  0.94x
  5    UNITDSPR  Rs94,569 Cr  Rs1302.90    -0.8%       8     8.1     8.1   52.8   19.2  1.25x  1.82x
  6    SUNPHARMA    Rs4.3L Cr  Rs1675.50    -3.0%      25     7.3     8.5   40.4   25.5  1.53x  2.35x
  7    HINDPETRO  Rs71,559 Cr  Rs 370.85    -3.5%      23     6.8     8.2   52.7   19.9  0.56x  0.90x
  8    RELIANCE   Rs19.1L Cr  Rs13

In [11]:
# ── CELL 6: Fundamental Deep Dive ─────────────────────────────
from difflib import get_close_matches

FUND_METRICS_CSV = os.path.join(DATA_DIR, 'fundamentals', 'fundamental_metrics_full.csv')
fund_metrics_df  = pd.read_csv(FUND_METRICS_CSV)

all_symbols = fund_metrics_df['Symbol'].tolist()

def validate_symbol(user_input):
    s = user_input.strip().upper()
    if s in all_symbols:
        return s
    matches = get_close_matches(s, all_symbols, n=3, cutoff=0.6)
    if not matches:
        print(f"  ❌ '{s}' not found. No close matches.")
        return None
    print(f"  ⚠️  '{s}' not found. Did you mean:")
    for i, m in enumerate(matches, 1):
        print(f"    {i}. {m}")
    print(f"    0. None / skip")
    choice = input("  Enter number: ").strip()
    if choice in [str(i) for i in range(1, len(matches)+1)]:
        return matches[int(choice)-1]
    return None

def na(val, suffix='', decimals=2, prefix=''):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return '—'
    try:
        return f"{prefix}{float(val):.{decimals}f}{suffix}"
    except:
        return str(val)

def flag(val, good_above=None, bad_below=None):
    try:
        v = float(val)
        if good_above is not None and v >= good_above: return '✅'
        if bad_below  is not None and v <= bad_below:  return '⚠️'
    except:
        pass
    return ''

def trend_arrow(val):
    try:
        v = float(val)
        if v > 0:  return f'▲{v:.2f}'
        elif v < 0: return f'▼{abs(v):.2f}'
        else:       return '→0.00'
    except:
        return '—'

def show_fundamental(symbol):
    # ── Get metrics row ───────────────────────────────────────
    m_row = fund_metrics_df[fund_metrics_df['Symbol'] == symbol]
    s_row = fund_df[fund_df['Symbol'] == symbol]
    t_row = work_df[work_df['Symbol'] == symbol]

    if len(m_row) == 0:
        print(f"  ❌ {symbol} not found in fundamental metrics")
        return
    m = m_row.iloc[0]
    s = s_row.iloc[0] if len(s_row) > 0 else None
    t = t_row.iloc[0] if len(t_row) > 0 else None

    sector   = m.get('Sector', '—')
    industry = m.get('Industry', '—')
    mcap     = mcap_str(t['Market Cap Cr']) if t is not None else '—'
    price    = f"Rs{t['Current Price']:.2f}" if t is not None else '—'
    cap_cat  = t['Cap Category'] if t is not None else '—'

    print(f"\n  {'═'*65}")
    print(f"  {symbol}  |  {sector}  |  {industry}")
    print(f"  {cap_cat}  |  MCap {mcap}  |  Price {price}")
    print(f"  {'═'*65}")

    # ── SCORES ────────────────────────────────────────────────
    if s is not None:
        print(f"\n  FUNDAMENTAL SCORES")
        print(f"  {'─'*65}")
        hist_s = s.get('Historical Score', '—')
        peer_s = s.get('Peer Score', '—')
        qual_s = s.get('Quality Score', '—')
        fin_s  = s.get('Final Score', '—')
        print(f"  Final Score      : {na(fin_s)}/100  "
              f"{'✅' if float(fin_s or 0) >= 65 else '⚠️' if float(fin_s or 0) < 50 else ''}")
        print(f"  Historical Score : {na(hist_s)}/40   "
              f"(Revenue CAGR + Profit CAGR + OPM + ROE)")
        print(f"  Peer Score       : {na(peer_s)}/40   "
              f"(vs sector peers on all metrics)")
        print(f"  Quality Score    : {na(qual_s)}/20   "
              f"(Promoter + FII/DII + CF + consistency)")

    # ── RELATIVE RANKING ──────────────────────────────────────
    if t is not None:
        print(f"\n  RELATIVE RANKING")
        print(f"  {'─'*65}")
        print(f"  Sector Rank  : {t['Sector Score']:.1f}/10  "
              f"(vs all {sector} stocks)")
        print(f"  Cap Rank     : {t['Cap Score']:.1f}/10  "
              f"(vs {cap_cat} {sector} stocks)")

    # ── GROWTH ────────────────────────────────────────────────
    print(f"\n  GROWTH")
    print(f"  {'─'*65}")
    print(f"  TTM Revenue      : Rs{na(m.get('TTM Revenue'))} Cr")
    print(f"  Revenue CAGR 5Y  : {na(m.get('Revenue CAGR 5Y'), suffix='%')}  "
          f"{flag(m.get('Revenue CAGR 5Y'), good_above=15, bad_below=5)}")
    print(f"  Revenue CAGR 10Y : {na(m.get('Revenue CAGR 10Y'), suffix='%')}")
    print(f"  Revenue YoY Q    : {na(m.get('Revenue YoY Q'), suffix='%')}  "
          f"{'Accelerating ✅' if m.get('Revenue Accelerating') == True else ''}")
    print(f"  Rev Consec YoY   : {na(m.get('Revenue Consecutive YoY'), decimals=0)} quarters")
    print(f"\n  TTM Profit       : Rs{na(m.get('TTM Profit'))} Cr")
    print(f"  Profit CAGR 5Y   : {na(m.get('Profit CAGR 5Y'), suffix='%')}  "
          f"{flag(m.get('Profit CAGR 5Y'), good_above=15, bad_below=5)}")
    print(f"  Profit CAGR 10Y  : {na(m.get('Profit CAGR 10Y'), suffix='%')}")
    print(f"  Profit YoY Q     : {na(m.get('Profit YoY Q'), suffix='%')}")
    print(f"  EPS Latest Q     : {na(m.get('Latest EPS Q'))}")
    print(f"  EPS YoY Growth   : {na(m.get('EPS YoY Growth'), suffix='%')}")

    # ── MARGINS & RETURNS ─────────────────────────────────────
    print(f"\n  MARGINS & RETURNS")
    print(f"  {'─'*65}")
    print(f"  OPM 5Y Avg       : {na(m.get('Avg OPM 5Y'), suffix='%')}  "
          f"{flag(m.get('Avg OPM 5Y'), good_above=20, bad_below=10)}")
    print(f"  OPM 10Y Avg      : {na(m.get('Avg OPM 10Y'), suffix='%')}")
    print(f"  OPM Latest Q     : {na(m.get('Latest OPM Q'), suffix='%')}  "
          f"{'Improving ✅' if m.get('Margin Improving') == True else 'Declining ⚠️' if m.get('Margin Improving') == False else ''}")
    print(f"  ROE              : {na(m.get('Final ROE'), suffix='%')}  "
          f"{flag(m.get('Final ROE'), good_above=15, bad_below=8)}")
    print(f"  ROCE Latest      : {na(m.get('Latest ROCE'), suffix='%')}  "
          f"{'Improving ✅' if m.get('ROCE Improving') == True else ''}")
    print(f"  ROCE 5Y Avg      : {na(m.get('Avg ROCE 5Y'), suffix='%')}")

    # ── BALANCE SHEET ─────────────────────────────────────────
    print(f"\n  BALANCE SHEET")
    print(f"  {'─'*65}")
    print(f"  Debt to Equity   : {na(m.get('Debt to Equity'))}  "
          f"{flag(m.get('Debt to Equity'), bad_below=0, good_above=-1)}  "
          f"{'Reducing ✅' if m.get('Debt Reducing') == True else 'Rising ⚠️' if m.get('Debt Reducing') == False else ''}")
    print(f"  Latest Debt      : Rs{na(m.get('Latest Debt'))} Cr")
    print(f"  Latest Equity    : Rs{na(m.get('Latest Equity'))} Cr")

    # ── CASH FLOW ─────────────────────────────────────────────
    print(f"\n  CASH FLOW")
    print(f"  {'─'*65}")
    cf_pos   = m.get('CF Positive Years', 0)
    cf_total = m.get('CF Total Years', 1)
    cf_pct   = round(float(cf_pos or 0) / float(cf_total or 1) * 100)
    print(f"  Operating CF     : Rs{na(m.get('Latest Operating CF'))} Cr  "
          f"{'Growing ✅' if m.get('CF Growing') == True else ''}")
    print(f"  CF Consistency   : {cf_pos:.0f}/{cf_total:.0f} years positive "
          f"({cf_pct}%)  "
          f"{flag(cf_pct, good_above=90, bad_below=70)}")

    # ── OWNERSHIP ─────────────────────────────────────────────
    print(f"\n  OWNERSHIP")
    print(f"  {'─'*65}")
    fii = float(m.get('FII Holding') or 0)
    dii = float(m.get('DII Holding') or 0)
    print(f"  Promoter Holding : {na(m.get('Promoter Holding'), suffix='%')}  "
          f"4Q: {trend_arrow(m.get('Promoter Change 4Q'))}  "
          f"8Q: {trend_arrow(m.get('Promoter Change 8Q'))}")
    print(f"  FII Holding      : {na(fii, suffix='%')}  "
          f"4Q: {trend_arrow(m.get('FII Change 4Q'))}")
    print(f"  DII Holding      : {na(dii, suffix='%')}  "
          f"4Q: {trend_arrow(m.get('DII Change 4Q'))}")
    print(f"  FII + DII Total  : {na(fii + dii, suffix='%')}")

    # ── TECHNICAL SNAPSHOT ────────────────────────────────────
    if t is not None:
        print(f"\n  TECHNICAL SNAPSHOT")
        print(f"  {'─'*65}")
        print(f"  ML Prediction    : {t.get('ML_Prediction','—')}  "
              f"({na(t.get('ML_Confidence'), suffix='%')})")
        print(f"  Best Setup       : {t.get('Best Setup','—')}")
        print(f"  Sector Trend     : {t.get('Sector Trend','—')}")
        print(f"  Forecast 25d     : {na(t.get('Forecast_25d_Pct'), suffix='%')}")
        print(f"  Forecast 45d     : {na(t.get('Forecast_45d_Pct'), suffix='%')}")
        print(f"  Forecast 180d    : {na(t.get('Forecast_180d_Pct'), suffix='%')}")

    print(f"\n  {'═'*65}")

# ── Interactive loop ───────────────────────────────────────────
print("\n" + "="*65)
print("  FUNDAMENTAL DEEP DIVE — Enter any stock symbol")
print("="*65)

while True:
    user_inp = input("\n  Symbol (or 'done'): ").strip()
    if user_inp.lower() == 'done':
        break
    sym = validate_symbol(user_inp)
    if sym:
        show_fundamental(sym)

print("\nCell 6 OK — Fundamental Deep Dive ready")


  FUNDAMENTAL DEEP DIVE — Enter any stock symbol



  Symbol (or 'done'):  infy



  ═════════════════════════════════════════════════════════════════
  INFY  |  Information Technology  |  Computers - Software & Consulting
  Large Cap  |  MCap Rs5.1L Cr  |  Price Rs1318.70
  ═════════════════════════════════════════════════════════════════

  FUNDAMENTAL SCORES
  ─────────────────────────────────────────────────────────────────
  Final Score      : 62.00/100  
  Historical Score : 24.00/40   (Revenue CAGR + Profit CAGR + OPM + ROE)
  Peer Score       : 26.00/40   (vs sector peers on all metrics)
  Quality Score    : 12.00/20   (Promoter + FII/DII + CF + consistency)

  RELATIVE RANKING
  ─────────────────────────────────────────────────────────────────
  Sector Rank  : 7.2/10  (vs all Information Technology stocks)
  Cap Rank     : 7.7/10  (vs Large Cap Information Technology stocks)

  GROWTH
  ─────────────────────────────────────────────────────────────────
  TTM Revenue      : Rs162990.00 Cr
  Revenue CAGR 5Y  : 12.41%  
  Revenue CAGR 10Y : 11.82%
  Revenue YoY


  Symbol (or 'done'):  done



Cell 6 OK — Fundamental Deep Dive ready


In [12]:
# ── CELL 7: Score & Rank Movers ───────────────────────────────

FUND_PREV_FILE = os.path.join(DATA_DIR, 'fundamentals', 'fundamental_scores_prev.csv')

if not os.path.exists(FUND_PREV_FILE):
    print("=" * 65)
    print("  SCORE & RANK MOVERS")
    print("=" * 65)
    print("""
  ⚠️  No previous quarter data found.
  
  This section compares current vs previous quarter to find:
    3A — Absolute Score change  (Final Score ▲▼)
    3B — Relative Rank change   (Sector Rank + Cap Rank ▲▼)

  To enable this section next quarter:
    1. Before running run_quarterly.py, copy:
       fundamental_scores_full.csv
         → fundamental_scores_prev.csv
    in the data/fundamentals/ folder.

  After that this section will automatically show:
    RELATIVELY STRONGER ✅  — rank improved even if score fell
    PEERS IMPROVED MORE ⚠️  — score rose but rank fell
  """)
    print("  Cell 7 OK — Will activate next quarter")

else:
    fund_prev_df = pd.read_csv(FUND_PREV_FILE)

    # ── Recompute Sector/Cap ranks for prev data ───────────────
    prefilt_df2 = pd.read_csv(PREFILT_FILE)
    fund_prev_df = fund_prev_df.merge(
        prefilt_df2[['Symbol', 'Market_Cap_Cr']], on='Symbol', how='left'
    )
    fund_prev_df['Cap Category'] = fund_prev_df['Market_Cap_Cr'].apply(classify_mcap)

    fund_prev_df['Prev Sector Score'] = 0.0
    for sector in fund_prev_df['Sector'].unique():
        mask      = fund_prev_df['Sector'] == sector
        max_score = fund_prev_df[mask]['Final Score'].max()
        if max_score > 0:
            fund_prev_df.loc[mask, 'Prev Sector Score'] = (
                fund_prev_df[mask]['Final Score'] / max_score * 10
            ).round(1)

    fund_prev_df['Prev Cap Score'] = 0.0
    for sector in fund_prev_df['Sector'].unique():
        for cap in CAP_ORDER:
            mask   = (fund_prev_df['Sector'] == sector) & \
                     (fund_prev_df['Cap Category'] == cap)
            subset = fund_prev_df[mask]
            if len(subset) == 0:
                continue
            max_score = subset['Final Score'].max()
            if max_score > 0:
                fund_prev_df.loc[mask, 'Prev Cap Score'] = (
                    subset['Final Score'] / max_score * 10
                ).round(1)

    # ── Merge current vs previous ──────────────────────────────
    compare_df = fund_df[['Symbol', 'Sector', 'Final Score',
                           'Sector Score', 'Cap Score']].merge(
        fund_prev_df[['Symbol', 'Final Score', 'Prev Sector Score', 'Prev Cap Score']]
        .rename(columns={'Final Score': 'Prev Final Score'}),
        on='Symbol', how='inner'
    )

    compare_df['Score Change']      = (compare_df['Final Score'] -
                                       compare_df['Prev Final Score']).round(1)
    compare_df['Sector Rank Change']= (compare_df['Sector Score'] -
                                       compare_df['Prev Sector Score']).round(1)
    compare_df['Cap Rank Change']   = (compare_df['Cap Score'] -
                                       compare_df['Prev Cap Score']).round(1)
    compare_df['Rank Sum Change']   = (compare_df['Sector Rank Change'] +
                                       compare_df['Cap Rank Change']).round(1)

    def verdict(row):
        sc  = row['Score Change']
        src = row['Sector Rank Change']
        crc = row['Cap Rank Change']
        if src > 0 and crc > 0:   return 'RELATIVELY STRONGER ✅'
        elif src < 0 and crc < 0: return 'RELATIVELY WEAKER ⚠️'
        elif src > 0 or crc > 0:  return 'MIXED →'
        else:                     return 'UNCHANGED →'

    compare_df['Verdict'] = compare_df.apply(verdict, axis=1)

    # ── 3A: Absolute Score Movers ──────────────────────────────
    print("=" * 80)
    print("  SCORE & RANK MOVERS — SECTION 3A: Absolute Score Change")
    print("  Note: Overall market may pull all scores down — use 3B for true signal")
    print("=" * 80)

    improved = compare_df[compare_df['Score Change'] > 0].sort_values(
        'Score Change', ascending=False).head(20)
    declined = compare_df[compare_df['Score Change'] < 0].sort_values(
        'Score Change').head(20)

    print(f"\n  TOP IMPROVERS (absolute score ▲)")
    print(f"  {'Symbol':<12} {'Sector':<25} {'Prev':>6}  {'Curr':>6}  {'Chg':>6}  Verdict")
    print(f"  {'─'*75}")
    for _, row in improved.iterrows():
        print(f"  {row['Symbol']:<12} {str(row['Sector']):<25} "
              f"{row['Prev Final Score']:>6.1f}  "
              f"{row['Final Score']:>6.1f}  "
              f"▲{row['Score Change']:>5.1f}  "
              f"{row['Verdict']}")

    print(f"\n  TOP DECLINERS (absolute score ▼)")
    print(f"  {'Symbol':<12} {'Sector':<25} {'Prev':>6}  {'Curr':>6}  {'Chg':>6}  Verdict")
    print(f"  {'─'*75}")
    for _, row in declined.iterrows():
        print(f"  {row['Symbol']:<12} {str(row['Sector']):<25} "
              f"{row['Prev Final Score']:>6.1f}  "
              f"{row['Final Score']:>6.1f}  "
              f"▼{abs(row['Score Change']):>5.1f}  "
              f"{row['Verdict']}")

    # ── 3B: Relative Rank Movers ───────────────────────────────
    print(f"\n{'='*80}")
    print(f"  SCORE & RANK MOVERS — SECTION 3B: Relative Rank Change")
    print(f"  Key insight: stock can fall in absolute score but rise in relative rank")
    print(f"{'='*80}")

    rel_improved = compare_df[compare_df['Rank Sum Change'] > 0].sort_values(
        'Rank Sum Change', ascending=False).head(20)
    rel_declined = compare_df[compare_df['Rank Sum Change'] < 0].sort_values(
        'Rank Sum Change').head(20)

    print(f"\n  RELATIVELY STRONGER (rank improved vs peers)")
    print(f"  {'Symbol':<12} {'Sector':<22} {'AbsScr':>7}  "
          f"{'SecRnk':>9}  {'CapRnk':>9}  Verdict")
    print(f"  {'─'*80}")
    for _, row in rel_improved.iterrows():
        sc_arrow  = f"▲{row['Score Change']:.1f}"  if row['Score Change'] > 0 \
                    else f"▼{abs(row['Score Change']):.1f}"
        src_arrow = f"▲{row['Sector Rank Change']:.1f}"
        crc_arrow = f"▲{row['Cap Rank Change']:.1f}"
        print(f"  {row['Symbol']:<12} {str(row['Sector']):<22} "
              f"{sc_arrow:>7}  "
              f"{row['Prev Sector Score']:.1f}→{row['Sector Score']:.1f}"
              f"({src_arrow}):>9  "
              f"{row['Prev Cap Score']:.1f}→{row['Cap Score']:.1f}"
              f"({crc_arrow}):>9  "
              f"{row['Verdict']}")

    print(f"\n  RELATIVELY WEAKER (rank declined vs peers)")
    print(f"  {'Symbol':<12} {'Sector':<22} {'AbsScr':>7}  "
          f"{'SecRnk':>9}  {'CapRnk':>9}  Verdict")
    print(f"  {'─'*80}")
    for _, row in rel_declined.iterrows():
        sc_arrow  = f"▲{row['Score Change']:.1f}"  if row['Score Change'] > 0 \
                    else f"▼{abs(row['Score Change']):.1f}"
        src_arrow = f"▼{abs(row['Sector Rank Change']):.1f}"
        crc_arrow = f"▼{abs(row['Cap Rank Change']):.1f}"
        print(f"  {row['Symbol']:<12} {str(row['Sector']):<22} "
              f"{sc_arrow:>7}  "
              f"{row['Prev Sector Score']:.1f}→{row['Sector Score']:.1f}"
              f"({src_arrow}):>9  "
              f"{row['Prev Cap Score']:.1f}→{row['Cap Score']:.1f}"
              f"({crc_arrow}):>9  "
              f"{row['Verdict']}")

    print(f"\n{'─'*80}")
    print(f"  AbsScr = absolute Final Score change")
    print(f"  SecRnk/CapRnk = relative rank vs peers (0-10)")
    print(f"  RELATIVELY STRONGER ✅ = both ranks improved")
    print(f"  PEERS IMPROVED MORE ⚠️  = both ranks declined")
    print(f"{'─'*80}")

    print("\nCell 7 OK — Score & Rank Movers ready")

  SCORE & RANK MOVERS

  ⚠️  No previous quarter data found.
  
  This section compares current vs previous quarter to find:
    3A — Absolute Score change  (Final Score ▲▼)
    3B — Relative Rank change   (Sector Rank + Cap Rank ▲▼)

  To enable this section next quarter:
    1. Before running run_quarterly.py, copy:
       fundamental_scores_full.csv
         → fundamental_scores_prev.csv
    in the data/fundamentals/ folder.

  After that this section will automatically show:
    RELATIVELY STRONGER ✅  — rank improved even if score fell
    PEERS IMPROVED MORE ⚠️  — score rose but rank fell
  
  Cell 7 OK — Will activate next quarter


In [17]:
# ── CELL 9: Add / Remove Watchlist ────────────────────────────
import csv
from difflib import get_close_matches

WATCHLIST_FILE = os.path.join(DATA_DIR, 'portfolio', 'reversal_watchlist.csv')
os.makedirs(os.path.join(DATA_DIR, 'portfolio'), exist_ok=True)

valid_symbols = set(work_df['Symbol'].tolist())

def validate_symbol_wl(user_input):
    s = user_input.strip().upper()
    if s in valid_symbols:
        return s
    matches = get_close_matches(s, list(valid_symbols), n=3, cutoff=0.6)
    if not matches:
        print(f"  '{s}' not found in universe. No close matches.")
        return None
    print(f"  '{s}' not found. Did you mean:")
    for i, m in enumerate(matches, 1):
        row = work_df[work_df['Symbol'] == m].iloc[0]
        print(f"    {i}. {m:<12} {row['Sector']:<25} "
              f"Rs{row['Current Price']:.2f}  "
              f"MCap {mcap_str(row['Market Cap Cr'])}")
    print(f"    0. None / skip")
    choice = input("  Enter number: ").strip()
    if choice in [str(i) for i in range(1, len(matches)+1)]:
        return matches[int(choice)-1]
    return None

def load_watchlist():
    if not os.path.exists(WATCHLIST_FILE):
        return pd.DataFrame(columns=['Symbol', 'Added_Date', 'Notes'])
    return pd.read_csv(WATCHLIST_FILE)

def save_watchlist(df):
    df.to_csv(WATCHLIST_FILE, index=False)
    print(f"  ✅ Watchlist saved — {len(df)} stocks")

def show_watchlist(wl_df):
    if len(wl_df) == 0:
        print(f"  Watchlist is empty.")
        return
    print(f"\n  {'#':<3}  {'Symbol':<10}  {'Sector':<25}  "
          f"{'Price':>8}  {'MCap':>12}  {'Setup':<12}  {'Added':>12}  Notes")
    print(f"  {'─'*105}")
    for i, row in wl_df.reset_index(drop=True).iterrows():
        sym  = str(row['Symbol'])
        t    = work_df[work_df['Symbol'] == sym]
        if len(t) > 0:
            tr    = t.iloc[0]
            sec   = str(tr['Sector'])[:24]
            price = f"Rs{tr['Current Price']:.2f}"
            mcap  = mcap_str(tr['Market Cap Cr'])
            setup = str(tr['Best Setup'])
        else:
            sec = price = mcap = setup = '—'
        print(f"  {i+1:<3}  {sym:<10}  {sec:<25}  "
              f"{price:>8}  {mcap:>12}  {setup:<12}  "
              f"{str(row['Added_Date']):>12}  "
              f"{str(row.get('Notes','') or '')}")

def run_watchlist_manager():
    while True:
        wl_df = load_watchlist()

        print(f"\n{'='*65}")
        print(f"  REVERSAL WATCHLIST — {len(wl_df)} stocks")
        print(f"{'='*65}")
        show_watchlist(wl_df)

        print(f"\n  Options:")
        print(f"  1. Add stocks")
        print(f"  2. Remove stocks")
        print(f"  3. Exit watchlist manager")
        sub = input("\n  Choice: ").strip()

        if sub == '1':
            print(f"\n  Enter symbols to add. Type 'done' when finished.")
            print(f"  (Any stock — bull or reversal candidate)")
            while True:
                user_inp = input("\n  Symbol (or 'done'): ").strip()
                if user_inp.lower() == 'done':
                    break
                sym = validate_symbol_wl(user_inp)
                if sym is None:
                    continue
                if sym in wl_df['Symbol'].values:
                    print(f"  ⚠️  {sym} already in watchlist.")
                    continue
                notes = input(f"  Notes for {sym} (optional): ").strip()
                new_row = pd.DataFrame([{
                    'Symbol'    : sym,
                    'Added_Date': today_str,
                    'Notes'     : notes,
                }])
                wl_df = pd.concat([wl_df, new_row], ignore_index=True)
                tr = work_df[work_df['Symbol'] == sym].iloc[0]
                print(f"  ✅ Added {sym} — {tr['Sector']} | "
                      f"Rs{tr['Current Price']:.2f} | "
                      f"MCap {mcap_str(tr['Market Cap Cr'])} | "
                      f"Setup: {tr['Best Setup']}")
            save_watchlist(wl_df)

        elif sub == '2':
            if len(wl_df) == 0:
                print(f"  Watchlist is empty.")
                continue
            to_remove = input(
                "\n  Row numbers to remove (comma separated): "
            ).strip()
            try:
                indices = [int(x.strip()) - 1 for x in to_remove.split(',')]
                removed = wl_df.iloc[indices]['Symbol'].tolist()
                wl_df   = wl_df.drop(wl_df.index[indices]).reset_index(drop=True)
                save_watchlist(wl_df)
                print(f"  ✅ Removed: {', '.join(removed)}")
            except:
                print(f"  Invalid input. No changes made.")

        elif sub == '3':
            print(f"  Exiting watchlist manager.")
            break
        else:
            print(f"  Invalid choice.")

    return load_watchlist()

# Run
wl_df = run_watchlist_manager()
print(f"\nCell 9 OK — {len(wl_df)} stocks in watchlist")


  REVERSAL WATCHLIST — 2 stocks

  #    Symbol      Sector                        Price          MCap  Setup                Added  Notes
  ─────────────────────────────────────────────────────────────────────────────────────────────────────────
  1    INFY        Information Technology     Rs1318.70     Rs5.1L Cr  Watching        2026-04-19  nan
  2    TCS         Information Technology     Rs2581.50     Rs8.6L Cr  Watching        2026-04-19  nan

  Options:
  1. Add stocks
  2. Remove stocks
  3. Exit watchlist manager



  Choice:  2

  Row numbers to remove (comma separated):  1


  ✅ Watchlist saved — 1 stocks
  ✅ Removed: INFY

  REVERSAL WATCHLIST — 1 stocks

  #    Symbol      Sector                        Price          MCap  Setup                Added  Notes
  ─────────────────────────────────────────────────────────────────────────────────────────────────────────
  1    TCS         Information Technology     Rs2581.50     Rs8.6L Cr  Watching        2026-04-19  nan

  Options:
  1. Add stocks
  2. Remove stocks
  3. Exit watchlist manager



  Choice:  3


  Exiting watchlist manager.

Cell 9 OK — 1 stocks in watchlist


In [18]:
# ── CELL 10: Watchlist Screen ──────────────────────────────────

def get_status(row):
    """Generate status message for NOT YET triggered stocks."""
    price  = float(row.get('Current Price', 0) or 0)
    ema50  = float(row.get('EMA50', 0) or 0)
    adx    = float(row.get('ADX', 0) or 0)
    rsi    = float(row.get('RSI', 0) or 0)
    rev    = float(row.get('Reversal Score', 0) or 0)
    setup  = str(row.get('Best Setup', ''))
    ml     = str(row.get('ML_Prediction', ''))

    if setup == 'Momentum' and ml == 'Bullish Continual':
        return 'Momentum confirmed ✅ — check LT portfolio'
    if price < ema50 and adx >= 30:
        return 'Downtrend still strong — wait (ADX high)'
    if price < ema50 and adx < 30 and rsi <= 35:
        return 'Deeply oversold — watch for bounce'
    if price < ema50 and adx < 30 and rsi > 35:
        return 'Basing below EMA50 — ADX fading'
    if price >= ema50 and rev < 50:
        return 'Above EMA50 — no reversal signal yet'
    if rev >= 50:
        return 'Reversal score building — watch closely'
    return 'Watching — no signal yet'

def run_watchlist_screen():
    wl_df = load_watchlist()

    print(f"\n{'='*65}")
    print(f"  WATCHLIST SCREEN — {len(wl_df)} stocks")
    print(f"{'='*65}")

    if len(wl_df) == 0:
        print(f"  Watchlist is empty. Add stocks via Section 4.")
        return

    # Classify each watchlist stock
    passed_1a = []
    passed_1b = []
    passed_1c = []
    passed_mom = []
    not_yet   = []

    for _, wrow in wl_df.iterrows():
        sym  = str(wrow['Symbol'])
        t    = work_df[work_df['Symbol'] == sym]
        if len(t) == 0:
            not_yet.append((sym, None))
            continue
        row = t.iloc[0]

        price  = float(row.get('Current Price', 0) or 0)
        ema50  = float(row.get('EMA50', 0) or 0)
        ema200 = float(row.get('EMA200', 0) or 0)
        adx    = float(row.get('ADX', 0) or 0)
        rsi    = float(row.get('RSI', 0) or 0)
        rev    = float(row.get('Reversal Score', 0) or 0)
        bot    = float(row.get('Bottom_Rev_Prob', 0) or 0)
        setup  = str(row.get('Best Setup', ''))
        ml     = str(row.get('ML_Prediction', ''))
        sec_r  = float(row.get('Sector Score', 0) or 0)
        cap_r  = float(row.get('Cap Score', 0) or 0)
        conf   = float(row.get('ML_Confidence', 0) or 0)

        # Check 1A
        if (setup == 'Reversal' and bot >= MIN_BOTTOM_PROB):
            passed_1a.append(row)
        # Check 1B
        elif (rev >= 50 and setup != 'Momentum'):
            passed_1b.append(row)
        # Check 1C
        elif (price < ema50 and setup == 'Watching'
              and adx < 30 and rsi > 35):
            passed_1c.append(row)
        # Check Momentum (bull stocks)
        elif (ml == 'Bullish Continual' and setup == 'Momentum'
              and price > ema50 > ema200):
            passed_mom.append(row)
        else:
            not_yet.append((sym, row))

    def print_table(rows, show_botprob=True, show_ema50pct=False):
        if show_ema50pct:
            print(f"\n  {'#':<3}  {'Symbol':<8}  {'MCap':>11}  {'Price':>8}  "
                  f"{'EMA50%':>7}  {'RevScr':>6}  {'SecRnk':>6}  {'CapRnk':>6}  "
                  f"{'RSI':>5}  {'ADX':>5}  {'V5D':>5}  {'VMax':>5}")
            print(f"  {'─'*106}")
        else:
            print(f"\n  {'#':<3}  {'Symbol':<8}  {'MCap':>11}  {'Price':>8}  "
                  f"{'BotProb':>7}  {'RevScr':>6}  {'SecRnk':>6}  {'CapRnk':>6}  "
                  f"{'RSI':>5}  {'ADX':>5}  {'V5D':>5}  {'VMax':>5}")
            print(f"  {'─'*110}")

        for i, row in enumerate(rows, 1):
            v5d  = float(row.get('Vol 5D Ratio', row.get('Vol Ratio', 0)) or 0)
            vmax = get_vmax(row['Symbol'])
            if show_ema50pct:
                price = float(row.get('Current Price', 0) or 0)
                ema50 = float(row.get('EMA50', 0) or 0)
                pct   = round((price - ema50) / ema50 * 100, 1) if ema50 > 0 else 0
                print(f"  {i:<3}  "
                      f"{row['Symbol']:<8}  "
                      f"{mcap_str(row['Market Cap Cr']):>11}  "
                      f"Rs{float(row['Current Price']):>7.2f}  "
                      f"{pct:>+6.1f}%  "
                      f"{float(row.get('Reversal Score',0)):>6.0f}  "
                      f"{float(row.get('Sector Score',0)):>6.1f}  "
                      f"{float(row.get('Cap Score',0)):>6.1f}  "
                      f"{float(row.get('RSI',0)):>5.1f}  "
                      f"{float(row.get('ADX',0)):>5.1f}  "
                      f"{v5d:>4.2f}x  "
                      f"{vmax:>4.2f}x")
            else:
                print(f"  {i:<3}  "
                      f"{row['Symbol']:<8}  "
                      f"{mcap_str(row['Market Cap Cr']):>11}  "
                      f"Rs{float(row['Current Price']):>7.2f}  "
                      f"{float(row.get('Bottom_Rev_Prob',0)):>6.1f}%  "
                      f"{float(row.get('Reversal Score',0)):>6.0f}  "
                      f"{float(row.get('Sector Score',0)):>6.1f}  "
                      f"{float(row.get('Cap Score',0)):>6.1f}  "
                      f"{float(row.get('RSI',0)):>5.1f}  "
                      f"{float(row.get('ADX',0)):>5.1f}  "
                      f"{v5d:>4.2f}x  "
                      f"{vmax:>4.2f}x")

    # ── PASSED 1A ─────────────────────────────────────────────
    print(f"\n  {'─'*65}")
    print(f"  ✅ PASSED — 1A ML Confirmed Reversal ({len(passed_1a)} stocks)")
    if passed_1a:
        print_table(passed_1a, show_botprob=True)
    else:
        print(f"  None currently.")

    # ── PASSED 1B ─────────────────────────────────────────────
    print(f"\n  {'─'*65}")
    print(f"  ✅ PASSED — 1B Technical Watch ({len(passed_1b)} stocks)")
    if passed_1b:
        print_table(passed_1b, show_botprob=True)
    else:
        print(f"  None currently.")

    # ── PASSED 1C ─────────────────────────────────────────────
    print(f"\n  {'─'*65}")
    print(f"  ✅ PASSED — 1C Basing Watch ({len(passed_1c)} stocks)")
    if passed_1c:
        print_table(passed_1c, show_botprob=False, show_ema50pct=True)
    else:
        print(f"  None currently.")

    # ── PASSED MOMENTUM ───────────────────────────────────────
    print(f"\n  {'─'*65}")
    print(f"  🚀 PASSED — Momentum (Bullish Continual + EMA OK) ({len(passed_mom)} stocks)")
    if passed_mom:
        print(f"\n  {'#':<3}  {'Symbol':<8}  {'MCap':>11}  {'Price':>8}  "
              f"{'Conf':>6}  {'SecRnk':>6}  {'CapRnk':>6}  "
              f"{'RSI':>5}  {'ADX':>5}  {'V5D':>5}  {'VMax':>5}")
        print(f"  {'─'*100}")
        for i, row in enumerate(passed_mom, 1):
            v5d  = float(row.get('Vol 5D Ratio', row.get('Vol Ratio', 0)) or 0)
            vmax = get_vmax(row['Symbol'])
            print(f"  {i:<3}  "
                  f"{row['Symbol']:<8}  "
                  f"{mcap_str(row['Market Cap Cr']):>11}  "
                  f"Rs{float(row['Current Price']):>7.2f}  "
                  f"{float(row.get('ML_Confidence',0)):>5.1f}%  "
                  f"{float(row.get('Sector Score',0)):>6.1f}  "
                  f"{float(row.get('Cap Score',0)):>6.1f}  "
                  f"{float(row.get('RSI',0)):>5.1f}  "
                  f"{float(row.get('ADX',0)):>5.1f}  "
                  f"{v5d:>4.2f}x  "
                  f"{vmax:>4.2f}x")
    else:
        print(f"  None currently.")

    # ── NOT YET TRIGGERED ─────────────────────────────────────
    print(f"\n  {'─'*65}")
    print(f"  ⏳ NOT YET TRIGGERED ({len(not_yet)} stocks)")
    if not_yet:
        print(f"\n  {'Symbol':<10}  {'RSI':>5}  {'ADX':>5}  "
              f"{'RevScr':>6}  {'EMA50%':>7}  {'Setup':<12}  Status")
        print(f"  {'─'*85}")
        for sym, row in not_yet:
            if row is None:
                print(f"  {sym:<10}  —  Not found in universe")
                continue
            price  = float(row.get('Current Price', 0) or 0)
            ema50  = float(row.get('EMA50', 0) or 0)
            pct    = round((price - ema50) / ema50 * 100, 1) if ema50 > 0 else 0
            print(f"  {sym:<10}  "
                  f"{float(row.get('RSI',0)):>5.1f}  "
                  f"{float(row.get('ADX',0)):>5.1f}  "
                  f"{float(row.get('Reversal Score',0)):>6.0f}  "
                  f"{pct:>+6.1f}%  "
                  f"{str(row.get('Best Setup','')):< 12}  "
                  f"{get_status(row)}")

    print(f"\n  {'─'*65}")
    print(f"  Summary: 1A={len(passed_1a)} | 1B={len(passed_1b)} | "
          f"1C={len(passed_1c)} | Momentum={len(passed_mom)} | "
          f"Waiting={len(not_yet)}")
    print(f"  {'─'*65}")

# Run
run_watchlist_screen()
print(f"\nCell 10 OK — Watchlist screen ready")


  WATCHLIST SCREEN — 1 stocks

  ─────────────────────────────────────────────────────────────────
  ✅ PASSED — 1A ML Confirmed Reversal (0 stocks)
  None currently.

  ─────────────────────────────────────────────────────────────────
  ✅ PASSED — 1B Technical Watch (0 stocks)
  None currently.

  ─────────────────────────────────────────────────────────────────
  ✅ PASSED — 1C Basing Watch (1 stocks)

  #    Symbol           MCap     Price   EMA50%  RevScr  SecRnk  CapRnk    RSI    ADX    V5D   VMax
  ──────────────────────────────────────────────────────────────────────────────────────────────────────────
  1    TCS         Rs8.6L Cr  Rs2581.50    -1.5%      30     7.2     7.7   55.6   24.3  1.21x  2.30x

  ─────────────────────────────────────────────────────────────────
  🚀 PASSED — Momentum (Bullish Continual + EMA OK) (0 stocks)
  None currently.

  ─────────────────────────────────────────────────────────────────
  ⏳ NOT YET TRIGGERED (0 stocks)

  ─────────────────────────────